# 04 — Embeddings & Vector Search

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives
1. Explain embeddings in plain language.
2. Compare **keyword search** with **semantic search**.
3. Chunk a document (size + overlap) and understand why.
4. Build a simple vector store over our PDFs.


## What is an embedding?

**Analogy.** An embedding turns a sentence into a *GPS coordinate of meaning*. Two sentences about the same topic land near each other, even if they use different words. We can then "search by meaning" instead of search by exact words.

Concretely: an embedding is a fixed-length vector of numbers (e.g. 384 numbers per chunk). We measure similarity with **cosine similarity** (dot-product after normalisation).

In [ ]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


## 4.1 — Keyword vs semantic search side by side

In [ ]:
import pandas as pd
from src.document_loaders import load_pdfs_in_folder
from src.rag_utils import VectorStore, chunk_documents

docs = load_pdfs_in_folder('data/generated/pdf')
print(f'Loaded {len(docs)} page-documents from PDFs')

In [ ]:
# Build the vector store (first run downloads the embedding model ~80 MB)
chunks = chunk_documents(docs, chunk_size=800, overlap=120)
print(f'Created {len(chunks)} chunks')
store = VectorStore()
store.add(chunks)
print(f'Vector store size: {len(store)} chunks')

In [ ]:
query = 'vendor approval thresholds'

# --- Keyword search (naive substring) ---
keyword_hits = [c for c in chunks if query.lower() in c['text'].lower()][:3]
print(f'\nKEYWORD SEARCH found {len(keyword_hits)} chunks')
for h in keyword_hits:
    print('  -', h['metadata'].get('source'), 'p.', h['metadata'].get('page'))

# --- Semantic search ---
semantic_hits = store.search(query, k=3)
print(f'\nSEMANTIC SEARCH top 3')
for h in semantic_hits:
    print(f"  - {h['metadata'].get('source'):40s} p.{h['metadata'].get('page')}  score={h['score']:.3f}")

**Why is semantic search different?** The exact phrase "vendor approval thresholds" may not appear anywhere — but the *Approval Matrix* in the Internal Control Policy and the *Vendor Selection* clause in the Procurement Policy are *semantically* close. Keyword search misses them; semantic search finds them.

## 4.2 — Inspect a chunk

In [ ]:
top = store.search('loan covenant compliance', k=1)[0]
print('Source:', top['metadata'])
print('Score :', round(top['score'], 3))
print('Text  :')
print(top['text'])

## 4.3 — Chunking — why size and overlap matter

In [ ]:
from src.rag_utils import chunk_text
long = docs[0]['text']
print('Document length (chars):', len(long))
for size in [200, 800, 1500]:
    cks = chunk_text(long, chunk_size=size, overlap=80)
    print(f'  chunk_size={size:5d}  →  {len(cks):3d} chunks  '
          f"avg {sum(len(c) for c in cks)//len(cks)} chars")

**Trade-off.** Small chunks = precise retrieval but lose context. Large chunks = more context but the LLM has to wade through irrelevant text. 600–1000 chars with 100–150 overlap is a sensible default for short business documents. Overlap prevents a fact from being split at the boundary.

## 4.4 — Filter by metadata

In [ ]:
# Search only inside the Procurement Policy
hits = store.search('sole-source approval', k=3, where={'source': '04_procurement_policy.pdf'})
for h in hits:
    print(h['metadata'], 'score=', round(h['score'], 3))
    print(h['text'][:200], '...\n')

## Expected output

* Keyword search → 0 hits for "vendor approval thresholds" (exact phrase isn't in the docs).
* Semantic search → top hits are the Internal Control Policy and Procurement Policy.


## Exercise

1. Try queries: `"related party benchmark"`, `"DSCR ratio"`, `"physical count exceptions"`. Inspect the top chunks. Are they the *right* ones?
2. Increase `chunk_size` to 2000 and re-build. Does retrieval get better or worse?


## Common errors

| Symptom | Fix |
|---|---|
| First run very slow | The embedding model (~80 MB) is downloading. Subsequent runs are fast. |
| Out-of-memory | Reduce chunks (larger `chunk_size`) or limit to a subset of PDFs. |
| Wrong chunk retrieved | Re-tune chunk size / overlap; consider a hybrid keyword+semantic search (out of scope today). |


## ⚠️ Professional caution

Semantic search gives you the **most similar** chunk — that is not the same as the **most authoritative** chunk. Citations matter (covered in Notebook 06).